# Assistant de recherche cosmétique en pharmacie

Ce notebook fournit un **prototype local** qui recherche des produits dans le catalogue Open Beauty Facts préparé dans `data_Open Beauty Facts/products_clean.csv`. Il n’utilise ni API externe, ni modèle de langage, ni connexion réseau.

**Limites importantes :** les résultats sont des candidats à examiner par un pharmacien, pas des recommandations médicales. Le catalogue peut être incomplet, bruité ou obsolète. Une composition absente ou un marqueur absent ne prouve jamais l’absence d’un ingrédient, une absence d’allergène ou la sécurité d’un produit. Vérifier l’emballage actuel et les sources professionnelles avant tout conseil.

**Utilisation :** exécuter les cellules dans l’ordre, puis lancer `chat()` dans la dernière cellule et saisir `quitter` pour terminer. Les exemples ci-dessous sont fictifs.

Source des données : [Open Beauty Facts](https://world.openbeautyfacts.org/data), base sous licence ODbL. Respecter l’attribution et les conditions de réutilisation de la base.

In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 240)
pd.set_option('display.max_columns', 20)
print('pandas', pd.__version__)

## Charger le catalogue local

Le chemin est recherché depuis le dossier courant et ses parents : le notebook peut donc être ouvert depuis la racine du dépôt ou depuis un sous-dossier.

In [ ]:
DATA_RELATIVE_PATH = Path('data_Open Beauty Facts') / 'products_clean.csv'
DATA_PATH = next(
    (base / DATA_RELATIVE_PATH for base in (Path.cwd(), *Path.cwd().parents)
     if (base / DATA_RELATIVE_PATH).is_file()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Catalogue introuvable. Placez le notebook dans le dépôt contenant '
        "'data_Open Beauty Facts/products_clean.csv'."
    )

catalogue = pd.read_csv(DATA_PATH, dtype=str, encoding='utf-8', encoding_errors='replace')
required_columns = {'product_name', 'ingredients_text'}
missing_columns = required_columns.difference(catalogue.columns)
if missing_columns:
    raise ValueError(f'Colonnes obligatoires absentes du catalogue : {sorted(missing_columns)}')

for column in catalogue.columns:
    catalogue[column] = catalogue[column].fillna('').astype(str)

print(f'Catalogue chargé : {len(catalogue):,} produits')
print(f'Fichier : {DATA_PATH}')
print('Colonnes :', ', '.join(catalogue.columns))
display(catalogue[['product_name', 'brands', 'categories_tags', 'ingredients_text']].head(5))

## Recherche explicable

Le score lexical privilégie les correspondances dans le nom et la marque, puis dans la catégorie et la composition. Il ne comprend pas les synonymes ni la morphologie : essayez plusieurs mots-clés. Le filtre de pays et l’exclusion d’un ingrédient servent uniquement à réduire la liste de candidats ; ils ne constituent pas une vérification d’allergie ou de sécurité.

In [ ]:
STOP_WORDS = {
    'avec', 'dans', 'pour', 'sans', 'une', 'des', 'les', 'sur', 'qui', 'que',
    'mon', 'ma', 'mes', 'du', 'de', 'la', 'le', 'un', 'et', 'est', 'au', 'aux',
    'je', 'veux', 'cherche', 'recherche', 'produit', 'peau', 'visage',
}

def normalize_text(value):
    """Lowercase and remove accents for approximate keyword matching."""
    text = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')
    return text.lower()


def tokens(value):
    return [word for word in re.findall(r'[a-z0-9]+', normalize_text(value))
            if len(word) > 1 and word not in STOP_WORDS]


def search_products(query, products=None, country=None, exclude_ingredients=None, top_k=5):
    """Return catalogue matches; an empty/unknown query never generates products."""
    products = catalogue if products is None else products
    query_tokens = list(dict.fromkeys(tokens(query)))
    if not query_tokens or products is None or products.empty:
        return pd.DataFrame()

    fields = {
        'product_name': 4,
        'brands': 3,
        'categories_tags': 3,
        'ingredients_text': 1,
        'ingredients_list_str': 1,
    }
    scored = []
    for _, row in products.iterrows():
        values = {name: str(row.get(name, '') or '') for name in fields}
        if country:
            country_value = normalize_text(row.get('countries_tags', ''))
            if normalize_text(country).strip() not in country_value:
                continue

        ingredient_text = normalize_text(
            values['ingredients_text'] + ' ' + values['ingredients_list_str']
        )
        exclusions = [normalize_text(item).strip() for item in (exclude_ingredients or []) if str(item).strip()]
        if any(item in ingredient_text for item in exclusions):
            continue

        score = 0
        for field, weight in fields.items():
            field_tokens = set(tokens(values[field]))
            score += weight * sum(token in field_tokens for token in query_tokens)
        if score:
            result = row.to_dict()
            result['match_score'] = score
            scored.append(result)

    if not scored:
        return pd.DataFrame()
    return (pd.DataFrame(scored)
            .sort_values(['match_score', 'product_name'], ascending=[False, True])
            .head(max(1, int(top_k)))
            .reset_index(drop=True))


def show_candidates(results):
    """Display candidates with provenance and explicit data-quality cautions."""
    if results.empty:
        print('Aucun résultat dans le catalogue pour ces mots-clés. Essayez une autre formulation ou vérifiez manuellement.')
        return

    for number, (_, product) in enumerate(results.iterrows(), start=1):
        name = product.get('product_name', '') or 'Nom non renseigné'
        brand = product.get('brands', '') or 'Marque non renseignée'
        print(f'\n{number}. {name} — {brand} (score lexical : {product["match_score"]})')
        print('Catégories :', product.get('categories_tags', '') or 'non renseignées')
        print('Composition fournie :', product.get('ingredients_text', '') or 'NON RENSEIGNÉE — impossible de conclure sur les ingrédients')
        print('Source produit :', product.get('url', '') or 'URL non renseignée')
        print('Marqueurs indicatifs dans le jeu de données :')
        print('  allergènes :', product.get('allergens_str', '') or 'aucun marqueur listé (cela ne prouve pas leur absence)')
        print('  marqueurs sensibles :', product.get('sensitive_flags_str', '') or 'aucun marqueur listé (liste indicative et incomplète)')
    print('\nÀ vérifier par le pharmacien sur l’emballage actuel. Ces résultats ne constituent pas un avis médical ni une garantie de sécurité.')


def recommend(query, country=None, exclude_ingredients=None, top_k=5):
    """Search and display candidate products for pharmacist review."""
    results = search_products(query, country=country,
                              exclude_ingredients=exclude_ingredients, top_k=top_k)
    show_candidates(results)
    return results

## Exemples de recherche (scénarios fictifs)

Ces exemples testent seulement la recherche textuelle du catalogue. Ils ne valident pas l’adéquation d’un produit pour une personne.

In [ ]:
# Exécuter un exemple à la fois et essayer d'autres mots-clés selon le catalogue.
recommend('crème hydratante visage', top_k=3)
# recommend('nettoyant peau sensible', country='france', top_k=3)

## Lancer le chatbot

Exécutez la cellule suivante puis saisissez une demande en français. Commandes disponibles : `aide`, `quitter`. Une demande peut être une recherche par mots-clés, par exemple `crème hydratante visage`. Le chatbot fait une recherche dans le catalogue ; il ne pose pas de diagnostic.

In [ ]:
def chat():
    """Start a simple interactive, local keyword-search chat session."""
    print('Assistant de recherche cosmétique — aide à la recherche, pas un avis médical.')
    print("Saisissez 'aide' pour les commandes ou 'quitter' pour terminer.\n")
    while True:
        try:
            message = input('Vous : ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nFin de la session.')
            break
        if normalize_text(message) in {'quitter', 'exit', 'stop'}:
            print('Fin de la session. Vérifiez les informations avec le pharmacien.')
            break
        if normalize_text(message) in {'aide', 'help'}:
            print('Décrivez le type de produit ou les ingrédients à rechercher. Exemples fictifs : crème hydratante visage ; nettoyant doux. Essayez plusieurs mots-clés si aucun résultat ne correspond.')
            continue
        if not tokens(message):
            print('Précisez quelques mots-clés pour rechercher dans le catalogue.')
            continue
        recommend(message, top_k=5)
        print('\nVous pouvez préciser votre recherche ou saisir une nouvelle demande.\n')

# Lancer interactivement en exécutant :
# chat()

## Vérifications rapides

Les tests utilisent un petit catalogue fictif pour vérifier le classement, la requête vide et la gestion de colonnes manquantes. Ils ne constituent pas une validation médicale.

In [ ]:
test_catalogue = pd.DataFrame([
    {'product_name': 'Crème hydratante visage', 'brands': 'Marque A',
     'categories_tags': 'soin visage', 'ingredients_text': 'aqua, glycerin',
     'url': 'https://example.invalid/a'},
    {'product_name': 'Shampooing', 'brands': 'Marque B',
     'categories_tags': 'cheveux', 'ingredients_text': 'aqua',
     'url': 'https://example.invalid/b'},
])

ranked = search_products('crème hydratante', products=test_catalogue)
assert not ranked.empty and ranked.iloc[0]['product_name'] == 'Crème hydratante visage'
assert search_products('   ', products=test_catalogue).empty
assert search_products('inconnu', products=test_catalogue).empty
assert search_products('hydratante', products=pd.DataFrame([{'product_name': 'Crème hydratante'}])).iloc[0]['product_name'] == 'Crème hydratante'
print('Tests réussis : classement lexical, requête vide, absence de résultat et champs facultatifs manquants.')

## Limites et prochaines étapes

- Le découpage et les marqueurs présents dans le jeu préparé sont indicatifs ; vérifier leur exactitude avant de les utiliser.
- La recherche ne comprend pas les synonymes, les fautes de frappe ni les besoins dermatologiques. Un résultat bien classé n’est pas une preuve d’adéquation.
- Un filtre d’ingrédient ne peut pas confirmer qu’un produit est exempt d’un ingrédient si la composition source est incomplète ou mal extraite.
- Avant tout usage réel : faire valider le catalogue et les règles par des pharmaciens, auditer les données, tester les abstentions, et examiner les obligations applicables en matière de santé, de confidentialité et de réglementation.

Le fichier brut `obf_raw.csv` n’est pas requis pour ce notebook : seul le catalogue nettoyé est utilisé.